# 프로토콜 v1.2 — WS3: 예측구간 보정 방식과 E2 정보 시점 규칙 (탐색 분석, CPU)

새 학습은 없습니다. 이미 저장된 예측값만 다시 계산합니다.

**WS3a — 보정 방식 비교 (v1.2 4.1절)**
- C0: v1.0 그대로, 전 서브셋 공통 c₉₀ (V0)
- C1: 서브셋별 c₉₀
- C2: 예측 중앙값 삼분위 구간별 c₉₀ (Mondrian)
- C3: 하한만 따로 보정 (실제 RUL < 하한 비율 5% 목표)

보정 상수는 모두 **개발 LOUO 예측값(Phase 1)**으로만 정합니다. 비적합 점수, 유한표본 보정, 서브셋 가중은 Phase 1b와 같습니다. 먼저 C0가 `calibration.json`의 값을 그대로 다시 만드는지 확인합니다.
- 개발 데이터 안에서는 한 엔진을 빼고 정한 상수로 그 엔진을 확인합니다(LOUO).
- 시험 데이터에는 v1.0 고정 예측값에 C0–C3를 적용합니다. 이미 본 데이터이므로 **기술 통계로만** 씁니다.

**WS3b — E2 정보 시점 규칙 (v1.2 4.2절)**
- R0: 사이클 t의 창에 t − 1 출력 − 1 (v1.1 규칙)
- R1: t − 1 출력 그대로
- R2: t 출력 그대로 (낙관적)

E2c CruiseBench, E2c bayesrul, E2b의 저장된 예측값으로 같은 창, 같은 붓스트랩을 다시 계산합니다. 먼저 R0가 저장된 v1.1 결과를 정확히 다시 만드는지 확인합니다.

**실행 전:** 이 노트북을 커밋한 주소를 `RUN_COMMIT_URL`에, v1.2 문서를 커밋한 주소를 `ADDENDUM_V12_URL`에 넣으세요. 런타임은 CPU, 고용량 RAM 권장입니다. 붓스트랩이 많아 30–60분 걸릴 수 있습니다.

In [ ]:
import os, sys, json, glob, time, hashlib, re, zipfile
import numpy as np, pandas as pd
from scipy.stats import norm
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/PI-BDP_NCMAPSS'; TMP = '/content/ws3'
else:
    ROOT = os.environ.get('NCM_ROOT', './ncm'); TMP = os.environ.get('WS3_TMP', './ws3_tmp')
P1 = f'{ROOT}/phase1'; IN3 = f'{ROOT}/phase3'; A11 = f'{ROOT}/addendum_v11'; OUT = f'{ROOT}/addendum_v12/ws3'
os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
URL = r'https://github\.com/.+/commit/[0-9a-f]{7,40}$'
ADDENDUM_V12_URL = ''   # v1.2 문서를 커밋한 주소
RUN_COMMIT_URL = ''     # 이 노트북을 커밋한 주소
assert re.match(URL, ADDENDUM_V12_URL) and re.match(URL, RUN_COMMIT_URL), '두 커밋 주소를 넣으세요.'
SUB9 = ['DS01-005', 'DS02-006', 'DS03-012', 'DS04', 'DS05', 'DS06', 'DS07', 'DS08a-009', 'DS08c-008']
CAPS = dict(zip(SUB9, [38, 34, 34, 43, 38, 41, 39, 34, 28]))
QL = [0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975]; QC = [f'q{int(round(q * 1000)):03d}' for q in QL]
K = ['subset', 'unit', 'cycle']; EPS = 0.5
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    return h.hexdigest()
REF3 = json.load(open(f'{IN3}/sha256_phase3.json'))
bad = [k for k, v in REF3.items() if os.path.exists(f'{IN3}/{k}') and sha(f'{IN3}/{k}') != v]; assert not bad, f'Phase 3 파일이 바뀜: {bad}'
cfg = json.load(open(f'{P1}/config.json')); CAL = json.load(open(f'{P1}/calibration.json'))
c90_ref = float(CAL['c']['90'])
V0 = pd.concat([pd.read_csv(f'{IN3}/test_PI-BDP_V0_{s}.csv.gz') for s in SUB9], ignore_index=True)
print('Phase 3 해시', len(REF3), '| c90 (v1.0)', c90_ref, '| 시험 사이클', len(V0))

## WS3a — 보정 방식 비교

In [ ]:
# === 개발 LOUO 예측값 (Phase 1b와 같은 선택) ===
Pr = pd.concat([pd.read_csv(f'{P1}/stageB_preds_{s}.csv.gz') for s in SUB9], ignore_index=True)
dev = Pr[(Pr.n_h == cfg['n_h']) & (Pr.shrink == cfg['shrink'])]
dev = dev[[k == cfg['k'][s] for s, k in zip(dev.subset, dev.k)]].reset_index(drop=True)

def score2(df):          # Phase 1b 비적합 점수 (90%)
    m, y = df.q500.values, df.true.values
    return np.maximum((m - y) / np.maximum(m - df.q050.values, EPS), (y - m) / np.maximum(df.q950.values - m, EPS))
def score_lo(df):        # 하한만
    m, y = df.q500.values, df.true.values
    return (m - y) / np.maximum(m - df.q050.values, EPS)
def wq(s, w, level):     # Phase 1b의 가중 conformal 분위수 (유한표본 보정)
    i = np.argsort(s); s, w = s[i], w[i] / w.sum()
    n = len(s); q = min(1.0, level * (n + 1) / n)
    return float(s[min(np.searchsorted(np.cumsum(w), q), n - 1)])
def wts(df): return 1.0 / df.groupby('subset').subset.transform('size').values

C0 = wq(score2(dev), wts(dev), 0.90)
assert abs(C0 - c90_ref) < 1e-9, f'C0 {C0}가 calibration.json {c90_ref}과 다름'
EDGES = np.quantile(dev.q500.values, [1 / 3, 2 / 3])          # C2 구간 경계 (개발 예측 중앙값의 삼분위)
def fit_all(d):
    out = dict(C0=wq(score2(d), wts(d), 0.90), C3=wq(score_lo(d), wts(d), 0.95))
    out['C1'] = {s: wq(score2(g), np.ones(len(g)), 0.90) for s, g in d.groupby('subset')}
    b = np.digitize(d.q500.values, EDGES)
    out['C2'] = {int(j): wq(score2(d[b == j]), wts(d[b == j]), 0.90) for j in range(3)}
    return out
CONST = fit_all(dev)
print('C0', round(CONST['C0'], 4), '| C3 (하한)', round(CONST['C3'], 4), '| C2', {k: round(v, 3) for k, v in CONST['C2'].items()})
print('C1', {k: round(v, 3) for k, v in CONST['C1'].items()})

def intervals(df, cs):
    m = df.q500.values; dl = np.maximum(m - df.q050.values, EPS); dh = np.maximum(df.q950.values - m, EPS)
    out = {'raw': (df.q050.values, df.q950.values)}
    out['C0'] = (np.maximum(m - cs['C0'] * dl, 0), m + cs['C0'] * dh)
    c1 = df.subset.map(cs['C1']).values; out['C1'] = (np.maximum(m - c1 * dl, 0), m + c1 * dh)
    c2 = pd.Series(np.digitize(m, EDGES)).map(cs['C2']).values; out['C2'] = (np.maximum(m - c2 * dl, 0), m + c2 * dh)
    out['C3'] = (np.maximum(m - cs['C3'] * dl, 0), out['C0'][1])      # 하한만 C3, 상한은 C0
    return out
def metrics_frame(df, ivs):
    y = df.true.values; rows = []
    for name, (L, U) in ivs.items():
        rows.append(pd.DataFrame({'method': name, 'subset': df.subset.values, 'life_bin': df.life_bin.values,
                                  'inside': ((y >= L) & (y <= U)).astype(float), 'below': (y < L).astype(float), 'width': U - L,
                                  'iscore': (U - L) + 20 * (L - y) * (y < L) + 20 * (y - U) * (y > U)}))
    return pd.concat(rows, ignore_index=True)
def life_bin(df): return pd.cut(df.cycle / (df.cycle + df.true), [0, .25, .5, .75, 1.0001], labels=['0-25%', '25-50%', '50-75%', '75-100%'], include_lowest=True)
AGG = dict(n=('inside', 'size'), cov90=('inside', 'mean'), below_L=('below', 'mean'), width90=('width', 'mean'), iscore90=('iscore', 'mean'))

In [ ]:
# === 개발 데이터 안의 확인: 한 엔진을 빼고 정한 상수로 그 엔진 (LOUO) ===
parts = []
for (s, u), g in dev.groupby(['subset', 'unit']):
    cs = fit_all(dev[~((dev.subset == s) & (dev.unit == u))])
    parts.append(metrics_frame(g.assign(life_bin=life_bin(g)), intervals(g, cs)))
DEVM = pd.concat(parts, ignore_index=True)
DEV_SUB = DEVM.groupby(['method', 'subset']).agg(**AGG).reset_index()
DEV_ALL = DEVM.groupby('method').agg(**AGG).reset_index()
DEV_SUB.to_csv(f'{OUT}/ws3a_dev_louo_subset.csv', index=False); DEV_ALL.to_csv(f'{OUT}/ws3a_dev_louo_all.csv', index=False)
print(DEV_ALL.round(3).to_string(index=False)); print(DEV_SUB.pivot(index='subset', columns='method', values='cov90').round(3).to_string())

In [ ]:
# === 시험 데이터 (v1.0 고정 예측값, 이미 본 자료 → 기술 통계) ===
TE = V0.assign(life_bin=life_bin(V0))
TESTM = metrics_frame(TE, intervals(TE, CONST))
TE_ALL = TESTM.groupby('method').agg(**AGG).reset_index()
TE_SUB = TESTM.groupby(['method', 'subset']).agg(**AGG).reset_index()
TE_BIN = TESTM.groupby(['method', 'life_bin'], observed=True).agg(**AGG).reset_index()
assert abs(TE_ALL.set_index('method').loc['C0', 'cov90'] - TESTM[TESTM.method == 'C0'].inside.mean()) < 1e-12
for nm, t in [('all', TE_ALL), ('subset', TE_SUB), ('lifebin', TE_BIN)]: t.to_csv(f'{OUT}/ws3a_test_{nm}.csv', index=False)
json.dump(dict(C0=CONST['C0'], C1=CONST['C1'], C2=CONST['C2'], C2_edges=EDGES.tolist(), C3_lower=CONST['C3']), open(f'{OUT}/ws3a_constants.json', 'w'), indent=1)
print(TE_ALL.round(3).to_string(index=False)); print(TE_SUB.pivot(index='subset', columns='method', values='cov90').round(3).to_string())
print(TE_BIN.pivot(index='life_bin', columns='method', values='cov90').round(3).to_string())

## WS3b — E2 정보 시점 규칙의 민감도

In [ ]:
# === 공통: 규칙별 PI-BDP 출력, 점수, 붓스트랩 (v1.1 채점 노트북과 같은 함수) ===
RULES = {'R0': (1, 1), 'R1': (1, 0), 'R2': (0, 0)}       # (몇 사이클 앞의 출력, 뺄 값)
def pibdp(B, rule):
    lag, minus = RULES[rule]
    src = V0.assign(cycle=V0.cycle + lag)[K + QC]
    X = B[K].merge(src, on=K, how='left'); assert X[QC].notna().all().all(), f'{rule}: PI-BDP 출력이 없는 창'
    return X[QC].values - minus
def pinball(Q, y):
    d = y[:, None] - Q; tau = np.array(QL)[None, :]
    return np.mean(np.maximum(tau * d, (tau - 1) * d), axis=1)
def iscore(L, U, y): return (U - L) + 20 * (L - y) * (y < L) + 20 * (y - U) * (y > U)
def pibdp_scores(Qraw, y, cap):
    clip = lambda v: np.clip(v, 0, cap[:, None] if np.ndim(v) == 2 else cap)
    m = Qraw[:, 4]; Lc = np.maximum(m - c90_ref * np.maximum(m - Qraw[:, 1], EPS), 0); Uc = m + c90_ref * np.maximum(Qraw[:, 7] - m, EPS)
    L, U = clip(Lc), clip(Uc)
    return clip(m), dict(iscore=iscore(L, U, y), cover=((y >= L) & (y <= U)).astype(float), pinball=pinball(clip(Qraw), y))
def hier_boot(ea, eb, subset, unit, kind, n_boot=100000, seed=1):
    va, vb = (ea ** 2, eb ** 2) if kind == 'rmse' else (ea, eb)
    U = pd.DataFrame({'subset': subset, 'unit': unit, 'a': va, 'b': vb}).groupby(['subset', 'unit']).agg(
        sa=('a', 'sum'), sb=('b', 'sum'), n=('a', 'size')).reset_index()
    subs = sorted(U.subset.unique()); rng = np.random.default_rng(seed)
    per = [U[U.subset == s][['sa', 'sb', 'n']].values for s in subs]
    SA = np.zeros(n_boot); SB = np.zeros(n_boot); N = np.zeros(n_boot)
    pick = rng.integers(0, len(subs), (n_boot, len(subs)))
    for j in range(len(subs)):
        for si, arr in enumerate(per):
            r = np.flatnonzero(pick[:, j] == si)
            if not len(r): continue
            tot = arr[rng.integers(0, len(arr), (len(r), len(arr)))].sum(1)
            SA[r] += tot[:, 0]; SB[r] += tot[:, 1]; N[r] += tot[:, 2]
    f = (lambda a, n: np.sqrt(a / n)) if kind == 'rmse' else (lambda a, n: a / n)
    d = f(SA, N) - f(SB, N); obs = float(f(U.sa.sum(), U.n.sum()) - f(U.sb.sum(), U.n.sum()))
    return dict(diff=obs, ci_lo=float(np.percentile(d, 2.5)), ci_hi=float(np.percentile(d, 97.5)),
                p_boot=float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))))
def wording(r):
    if r['ci_lo'] > 0: return 'PI-BDP less accurate'
    if r['ci_hi'] < 0: return 'PI-BDP more accurate (exploratory)'
    return 'comparable'
def rmse(e): return float(np.sqrt(np.mean(np.square(e))))
RES = []
def run_rules(study, B, comp_pt, comp_pr):
    # comp_pt: {model: point array}, comp_pr: {model: {'iscore': array}} or {} (점 예측만인 경우)
    y, cap = B.label.values, B.cap.values
    for rule in RULES:
        mpt, mpr = pibdp_scores(pibdp(B, rule), y, cap)
        for mdl, p in comp_pt.items():
            r = hier_boot(mpt - y, p - y, B.subset.values, B.unit.values, 'rmse')
            RES.append(dict(study=study, rule=rule, metric='pooled RMSE', comparison=f'V0 - {mdl}', v0=rmse(mpt - y), model=rmse(p - y), **r, wording=wording(r)))
            if mdl in comp_pr:
                r = hier_boot(mpr['iscore'], comp_pr[mdl]['iscore'], B.subset.values, B.unit.values, 'mean')
                RES.append(dict(study=study, rule=rule, metric='interval score (90%)', comparison=f'V0 (calibrated) - {mdl}',
                                v0=float(mpr['iscore'].mean()), model=float(comp_pr[mdl]['iscore'].mean()), **r, wording=wording(r)))
        print(study, rule, '완료')
def gauss_scores(mu, sd, y, cap):
    Q = np.clip(mu[:, None] + sd[:, None] * norm.ppf(QL)[None, :], 0, cap[:, None])
    L, U = Q[:, 1], Q[:, 7]
    return dict(iscore=iscore(L, U, y), cover=((y >= L) & (y <= U)).astype(float), pinball=pinball(Q, y))

In [ ]:
# === E2c CruiseBench (점 예측만) ===
RC = f'{A11}/e2c_cruise'; NS = json.load(open(f'{RC}/seed_decision.json'))['n_seeds']; KW = ['subset', 'unit', 'cycle', 'win']
P = {m: pd.concat([pd.read_csv(f'{RC}/preds/{s}_{m}_s{sd}.csv.gz') for s in SUB9 for sd in range(1, NS + 1)], ignore_index=True)
     for m in ['LSTM', 'GRU', 'TCN', 'TSMixer']}
ref = P['TSMixer'][P['TSMixer'].seed == 1][KW + ['label_capped']].sort_values(KW).reset_index(drop=True)
first = ref.cycle == ref.groupby(['subset', 'unit']).cycle.transform('min')
B = ref[~first].reset_index(drop=True).rename(columns={'label_capped': 'label'}); B['cap'] = B.subset.map(CAPS)
pt = {}
for m, df in P.items():
    x = B[KW].merge(df.groupby(KW, as_index=False).pred.mean(), on=KW, how='left'); assert x.pred.notna().all()
    pt[m] = np.clip(x.pred.values, 0, B.cap.values)
run_rules('E2c CruiseBench', B, pt, {}); del P

In [ ]:
# === E2c bayesrul ===
RB = f'{A11}/e2c_bayesrul'
W = pd.read_csv(f'{RB}/windows_main.csv.gz'); W['subset'] = W.file.str.replace('N-CMAPSS_', '', regex=False)
W['cap'] = W.groupby(['subset', 'unit']).label.transform('max')
first = W.cycle == W.groupby(['subset', 'unit']).cycle.transform('min')
B = W[~first].reset_index(drop=True); IDX = B.idx.values; y, cap = B.label.values, B.cap.values
pt, pr = {}, {}
for meth in ['HNN', 'MCD', 'DE']:
    runs = [pd.read_csv(f).set_index('idx').loc[IDX] for f in sorted(glob.glob(f'{RB}/preds/main_{meth}_[0-9][0-9][0-9].csv.gz'))]
    pt[meth] = np.clip(np.mean([d.preds.values for d in runs], axis=0), 0, cap)
    sc = [gauss_scores(d.preds.values, d.stds.values, y, cap) for d in runs]
    pr[meth] = {k: np.mean([s[k] for s in sc], axis=0) for k in sc[0]}
run_rules('E2c bayesrul', B, pt, pr)

In [ ]:
# === E2b figshare (DS01–DS05) ===
zips = sorted(glob.glob(f'{A11}/e2b_figshare/*.zip')); assert len(zips) == 1
with zipfile.ZipFile(zips[0]) as z:
    mem = [n for n in z.namelist() if n.replace('\\', '/').endswith('cache/test/predictions.parquet')]; PQ = z.extract(mem[0], TMP)
F = pd.read_parquet(PQ, columns=['model', 'method', 'ds_id', 'unit_id', 'win_id', 'labels', 'preds', 'stds']).drop_duplicates(['model', 'ds_id', 'unit_id', 'win_id'])
F = F.rename(columns={'unit_id': 'unit'}); KEY = ['ds_id', 'unit', 'win_id']
W = pd.read_csv(f'{RB}/windows_rep.csv.gz'); W['subset'] = W.file.str.replace('N-CMAPSS_', '', regex=False)
W['cap'] = W.groupby(['subset', 'unit']).label.transform('max')
first = W.cycle == W.groupby(['subset', 'unit']).cycle.transform('min')
B = W[~first].sort_values(KEY).reset_index(drop=True); y, cap = B.label.values, B.cap.values
pt, pr = {}, {}
for meth in ['HNN', 'MCD', 'DE', 'LRT', 'FO', 'RAD']:
    mus, sc = [], []
    for mm in sorted(F[F.method == meth].model.unique()):
        d = B[KEY].merge(F[F.model == mm][KEY + ['preds', 'stds']], on=KEY, how='left'); assert d.preds.notna().all()
        mus.append(d.preds.values.astype(float)); sc.append(gauss_scores(d.preds.values.astype(float), d.stds.values.astype(float), y, cap))
    pt[meth] = np.clip(np.mean(mus, axis=0), 0, cap); pr[meth] = {k: np.mean([s[k] for s in sc], axis=0) for k in sc[0]}
run_rules('E2b figshare', B, pt, pr); del F

In [ ]:
# === R0가 v1.1 저장 결과를 다시 만드는지 확인, 표현이 바뀌는 비교 ===
R = pd.DataFrame(RES); R.to_csv(f'{OUT}/ws3b_rules.csv', index=False)
saved = {'E2c CruiseBench': f'{A11}/e2c_tests.csv', 'E2c bayesrul': f'{A11}/e2c_br_tests.csv', 'E2b figshare': f'{A11}/e2b_tests.csv'}
chk = []
for study, f in saved.items():
    S = pd.read_csv(f); S['metric'] = S['metric'] if 'metric' in S else 'pooled RMSE'
    X = R[(R.study == study) & (R.rule == 'R0')].merge(S, on=['metric', 'comparison'], suffixes=('', '_v11'))
    assert len(X) == len(S), f'{study}: 비교 수가 다름'
    ok = np.allclose(X[['diff', 'ci_lo', 'ci_hi']].values, X[['diff_v11', 'ci_lo_v11', 'ci_hi_v11']].values, atol=1e-9)
    assert ok, f'{study}: R0가 v1.1 결과와 다름'; chk.append(study)
print('R0 = v1.1 저장 결과 (정확히 같음):', chk)
WD = R.pivot_table(index=['study', 'metric', 'comparison'], columns='rule', values='wording', aggfunc='first').reset_index()
WD['changed'] = (WD.R1 != WD.R0) | (WD.R2 != WD.R0); WD.to_csv(f'{OUT}/ws3b_wording.csv', index=False)
print(WD.to_string(index=False))

In [ ]:
# === 요약 + 해시 ===
txt = '\n'.join([f"Addendum v1.2 WS3 (exploratory) {time.strftime('%Y-%m-%d %H:%M:%S')}  addendum={ADDENDUM_V12_URL}  run={RUN_COMMIT_URL}",
    '', f"WS3a constants (dev LOUO predictions only): C0={CONST['C0']:.4f} (= v1.0), C3 lower={CONST['C3']:.4f}, C2 edges={np.round(EDGES, 2).tolist()} -> {[round(v, 3) for v in CONST['C2'].values()]}",
    'C1 by subset: ' + json.dumps({k: round(v, 3) for k, v in CONST['C1'].items()}),
    '', 'WS3a dev LOUO check (all):', DEV_ALL.round(3).to_string(index=False),
    '', 'WS3a dev LOUO coverage by subset:', DEV_SUB.pivot(index='subset', columns='method', values='cov90').round(3).to_string(),
    '', 'WS3a test (already-seen v1.0 predictions; descriptive):', TE_ALL.round(3).to_string(index=False),
    '', 'WS3a test coverage by subset:', TE_SUB.pivot(index='subset', columns='method', values='cov90').round(3).to_string(),
    '', 'WS3a test fraction below lower bound by subset:', TE_SUB.pivot(index='subset', columns='method', values='below_L').round(3).to_string(),
    '', 'WS3a test coverage by life bin:', TE_BIN.pivot(index='life_bin', columns='method', values='cov90').round(3).to_string(),
    '', f'WS3b information-timing rules; R0 reproduces v1.1 exactly for {chk}:', R.round(4).to_string(index=False),
    '', 'WS3b wording by rule:', WD.to_string(index=False)])
open(f'{OUT}/ws3_summary.txt', 'w').write(txt)
H = {os.path.basename(p): sha(p) for p in sorted(glob.glob(f'{OUT}/ws3*.*'))}
json.dump(H, open(f'{OUT}/sha256_ws3.json', 'w'), indent=1); print(txt)